# פרויקט מדעי הנתונים – חיזוי נשירת סטודנטים
## הכפר הירוק, כיתה י'

**שם התלמיד:** עידן עטר | ת"ז: 318074846

**שם המורה:** ענת שפיר

**מטרת המחקר:** לחזות אם תלמיד נשר מאוניברסיטה או לא, באמצעות מודלים של למידת מכונה.

**מקור הנתונים:** [Kaggle – Student Dropout Prediction Dataset](https://www.kaggle.com/datasets/meharshanali/student-dropout-prediction-dataset)

## פירוט על המאגר

המאגר מכיל מידע על סטודנטים ונשירתם מאוניברסיטה. להלן העמודות:

- **Student_ID:** מזהה סטודנט ייחודי.
- **Age:** גיל הסטודנט בשנים.
- **Gender:** Male / Female (מין).
- **Family_Income:** הכנסה משפחתית (Low / Medium / High).
- **GPA:** ציון ממוצע (0–4).
- **Attendance_Rate:** אחוז נוכחות (0–100).
- **Study_Hours_per_Day:** שעות לימוד יומיות.
- **Assignment_Delay_Days:** ימי עיכוב בהגשות.
- **Stress_Index:** מדד סטרס (0–1).
- **Dropout:** משתנה מטרה – 1 = נשר, 0 = לא נשר.

## חלק א' – טעינת ספריות

בבלוק הבא אנחנו מייבאים את כל הספריות שנצטרך לאורך הפרויקט:
- **pandas** ו-**numpy** – לעיבוד ולניתוח נתונים.
- **matplotlib** ו-**seaborn** – להצגת גרפים ותרשימים.
- **sklearn** – ספריית למידת מכונה: חלוקה, נרמול, מודלים, ומדדי הערכה.
- **imblearn** – לטיפול בחוסר איזון בנתונים באמצעות SMOTE.
- **kagglehub** – להורדת מאגר הנתונים ישירות מ-Kaggle.

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import warnings
warnings.filterwarnings('ignore')
import itertools
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from imblearn.over_sampling import SMOTE
from collections import Counter
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV

## טעינת הנתונים

בבלוק הבא אנחנו מורידים את מאגר הנתונים מ-Kaggle ומטעינים אותו לטבלה (DataFrame).
הפקודה `kagglehub.dataset_download` מורידה את הקובץ, ואנחנו מוצאים את קובץ ה-CSV ומטעינים אותו עם `pd.read_csv`.
לבסוף מדפיסים את 5 השורות הראשונות כדי לוודא שהנתונים נטענו כראוי.

In [ ]:
path = kagglehub.dataset_download("meharshanali/student-dropout-prediction-dataset")
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.read_csv(os.path.join(path, csv_files[0]))

print("הנתונים נטענו בהצלחה!")
print(f"מספר שורות: {df.shape[0]}, מספר עמודות: {df.shape[1]}")
print()
df.head()

## סקירה ראשונית של הנתונים (EDA)

לפני שמתחילים לעבד את הנתונים, חשוב להכיר אותם:
- **df.info()** – מראה את סוגי העמודות, כמה ערכים לא ריקים יש בכל עמודה, וצריכת הזיכרון. זה עוזר לנו להבין אם יש עמודות שחסרים בהן ערכים ומה הטיפוס של כל עמודה.
- **df.describe()** – מחשב סטטיסטיקות בסיסיות (ממוצע, חציון, סטיית תקן, מינימום, מקסימום) לכל עמודה מספרית. זה עוזר לזהות טווחי ערכים חריגים.
- **df.duplicated()** – בודק אם יש שורות כפולות במאגר, כי כפילויות עלולות להטות את המודל.

In [ ]:
print("=" * 60)
print("מבנה הנתונים - df.info()")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("סטטיסטיקות תיאוריות - df.describe()")
print("=" * 60)
df.describe()

In [ ]:
print("=" * 60)
print("סטטיסטיקות לעמודות קטגוריאליות")
print("=" * 60)
df.describe(include='object')

### בדיקת כפילויות

בודקים אם יש שורות זהות לחלוטין. שורות כפולות עלולות להטות את תוצאות המודל כי הן מכפילות דוגמאות זהות.

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"מספר שורות כפולות: {num_duplicates}")

if num_duplicates > 0:
    df = df.drop_duplicates()
    print(f"שורות אחרי הסרת כפילויות: {len(df)}")
else:
    print("אין שורות כפולות – הנתונים נקיים מכפילויות.")

### בדיקת ערכים חסרים

ערכים חסרים (NaN) עלולים לגרום לשגיאות באימון המודל. בודקים כמה ערכים חסרים יש בכל עמודה ומחליטים איך לטפל בהם.

In [ ]:
print("ערכים חסרים לפי עמודה:")
print(df.isnull().sum())
print(f"\nסך הכל שורות עם ערך חסר: {df.isnull().any(axis=1).sum()}")
print(f"אחוז השורות עם ערך חסר: {df.isnull().any(axis=1).mean():.1%}")

### המרת משתנים קטגוריאליים

חלק מהעמודות במאגר הן קטגוריאליות (טקסט), כמו Gender ו-Family_Income.
מודלים של למידת מכונה לא יכולים לעבוד עם טקסט, ולכן צריך להמיר אותן למספרים.

נשתמש ב-**LabelEncoder** שממיר כל קטגוריה למספר. לדוגמה:
- Gender: Male → 1, Female → 0
- Family_Income: Low → 1, Medium → 2, High → 0

חשוב לעשות את ההמרה לפני שמוחקים שורות, כדי לא לאבד מידע שימושי.

In [ ]:
label_encoders = {}
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("עמודות קטגוריאליות שנמצאו:", categorical_cols)
print()

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nכל המשתנים הקטגוריאליים הומרו למספרים בהצלחה.")

### ניקוי ערכים חסרים

לאחר ההמרה, מסירים שורות שנותרו עם ערכים חסרים. האחוז נמוך ולכן ההסרה לא תשפיע מהותית על הנתונים.

In [ ]:
print(f"שורות לפני ניקוי: {len(df)}")
df_cleaned = df.dropna()
print(f"שורות אחרי ניקוי: {len(df_cleaned)}")
print(f"שורות שהוסרו: {len(df) - len(df_cleaned)}")

### בדיקת חריגים (Outliers)

חריגים הם ערכים קיצוניים שנמצאים רחוק מרוב הנתונים. הם עלולים להטעות את המודל.
נשתמש בשיטת **IQR (Interquartile Range)**:
- מחשבים את הרבעון הראשון (Q1) והשלישי (Q3).
- IQR = Q3 - Q1
- ערך חריג הוא כל ערך שנמוך מ-Q1 - 1.5×IQR או גבוה מ-Q3 + 1.5×IQR.

במקום למחוק חריגים (מה שעלול להקטין מאוד את המאגר), נבצע **חיתוך (clipping)** – נעגל את הערכים הקיצוניים לגבולות.

In [ ]:
numeric_cols = df_cleaned.select_dtypes(include=['number']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['Student_ID', 'Dropout']]

print("בדיקת חריגים בעמודות מספריות (שיטת IQR):")
print("-" * 50)

outlier_counts = {}
for col in numeric_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_cleaned[col] < lower) | (df_cleaned[col] > upper)).sum()
    outlier_counts[col] = n_outliers
    if n_outliers > 0:
        print(f"  {col}: {n_outliers} חריגים (טווח תקין: {lower:.2f} – {upper:.2f})")

if sum(outlier_counts.values()) == 0:
    print("  לא נמצאו חריגים בשום עמודה.")
else:
    print(f"\nסך הכל חריגים: {sum(outlier_counts.values())}")
    print("מבצעים clipping – חיתוך ערכים קיצוניים לגבולות:")
    for col in numeric_cols:
        Q1 = df_cleaned[col].quantile(0.25)
        Q3 = df_cleaned[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_cleaned[col] = df_cleaned[col].clip(lower, upper)
    print("  החיתוך בוצע בהצלחה.")

## חלק ב' – ויזואליזציה וניתוח גרפי

### גרף 1: מפה תרמית (Heatmap) של קורלציות

מפה תרמית מראה את הקשר (קורלציה) בין כל זוג משתנים מספריים. ערך קרוב ל-1 או ל-(-1) אומר שיש קשר חזק בין שני המשתנים, וערך קרוב ל-0 אומר שאין קשר. זה עוזר לנו לבחור את הפיצ'רים הכי רלוונטיים לחיזוי נשירה.

In [ ]:
numeric_df = df_cleaned.select_dtypes(include=['number'])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix,
            annot=True,
            fmt=".2f",
            cmap='RdBu_r',
            center=0,
            linewidths=0.5,
            square=True)

plt.title("מפה תרמית של קורלציות בין כל המשתנים", fontsize=16)
plt.tight_layout()
plt.show()

### גרף 2: תרשים עמודות (Bar Chart) – שיעור נשירה לפי קטגוריות

תרשים עמודות מאפשר להשוות בין קבוצות שונות. נבדוק את שיעור הנשירה לפי מין (Gender) ולפי רמת הכנסה משפחתית (Family_Income), כדי לראות האם יש הבדלים משמעותיים בין הקבוצות.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dropout rate by Gender
if 'Gender' in label_encoders:
    le_gender = label_encoders['Gender']
    gender_labels = le_gender.classes_
    gender_dropout = df_cleaned.groupby('Gender')['Dropout'].mean() * 100
    axes[0].bar([gender_labels[i] for i in gender_dropout.index], gender_dropout.values, color=['#4C72B0', '#DD8452'])
    axes[0].set_title('שיעור נשירה לפי מין', fontsize=14)
    axes[0].set_ylabel('אחוז נשירה (%)')
    axes[0].set_xlabel('מין')
    for i, v in enumerate(gender_dropout.values):
        axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Dropout rate by Family_Income
if 'Family_Income' in label_encoders:
    le_income = label_encoders['Family_Income']
    income_labels = le_income.classes_
    income_dropout = df_cleaned.groupby('Family_Income')['Dropout'].mean() * 100
    axes[1].bar([income_labels[i] for i in income_dropout.index], income_dropout.values, color=['#55A868', '#C44E52', '#8172B2'])
    axes[1].set_title('שיעור נשירה לפי הכנסה משפחתית', fontsize=14)
    axes[1].set_ylabel('אחוז נשירה (%)')
    axes[1].set_xlabel('הכנסה משפחתית')
    for i, v in enumerate(income_dropout.values):
        axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### גרף 3: תרשים קופסה (Box Plot) – התפלגות משתנים לפי סטטוס נשירה

Box Plot מציג את ההתפלגות של כל משתנה: החציון, הרבעונים, הטווח, ואת הערכים החריגים. נשווה את ההתפלגויות בין סטודנטים שנשרו לכאלה שלא, כדי לראות אילו משתנים שונים משמעותית בין שתי הקבוצות.

In [ ]:
plot_features = ['GPA', 'Attendance_Rate', 'Study_Hours_per_Day', 'Assignment_Delay_Days', 'Stress_Index']

fig, axes = plt.subplots(1, len(plot_features), figsize=(18, 5))

for i, col in enumerate(plot_features):
    sns.boxplot(x='Dropout', y=col, data=df_cleaned, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col}', fontsize=11)
    axes[i].set_xlabel('נשירה (0=לא, 1=כן)')

plt.suptitle('התפלגות משתנים לפי סטטוס נשירה – Box Plot', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### בחירת פיצ'רים (Feature Selection)

לפי המפה התרמית, הערכים בעלי הקורלציה הכי גבוהה עם Dropout הם:
Stress_Index, GPA, Assignment_Delay_Days, Attendance_Rate, Study_Hours_per_Day.

נבדוק את כל הצירופים האפשריים שלהם כדי למצוא את קבוצת הפיצ'רים שנותנת את התוצאה הטובה ביותר.
חשוב: החלוקה ל-train/test נעשית **לפני** בחירת הפיצ'רים, כדי למנוע דליפת מידע (data leakage).

In [ ]:
CANDIDATE_FEATURES = [
    "Stress_Index",
    "GPA",
    "Assignment_Delay_Days",
    "Attendance_Rate",
    "Study_Hours_per_Day",
]

TARGET = "Dropout"

x_all = df_cleaned[CANDIDATE_FEATURES]
y_all = df_cleaned[TARGET].values

x_train_fs, x_test_fs, y_train_fs, y_test_fs = train_test_split(
    x_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f"Train size: {len(x_train_fs)}, Test size: {len(x_test_fs)}")
print(f"בחירת פיצ'רים תתבצע רק על סט האימון (ללא דליפה).")

subsets = []
for size in range(2, len(CANDIDATE_FEATURES) + 1):
    for combo in itertools.combinations(CANDIDATE_FEATURES, size):
        subsets.append(list(combo))

print(f"מספר צירופים לבדיקה: {len(subsets)}")
print("=" * 65)

SMOTE_RATIO  = 0.6
CV_FOLDS     = 5
RANDOM_STATE = 42

cv_combo = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

combo_results = []

for combo in subsets:
    X_combo = x_train_fs[combo].values

    pipe = ImbPipeline([
        ("smote",  SMOTE(sampling_strategy=SMOTE_RATIO, random_state=RANDOM_STATE)),
        ("scaler", MinMaxScaler()),
        ("clf",    SVC(kernel="rbf", C=10, random_state=RANDOM_STATE)),
    ])

    scores = cross_val_score(pipe, X_combo, y_train_fs,
                             cv=cv_combo, scoring="f1_macro", n_jobs=-1)

    combo_results.append({
        "features":  combo,
        "f1_mean":   scores.mean(),
        "f1_std":    scores.std(),
        "n_features": len(combo),
    })

combo_results.sort(key=lambda r: r["f1_mean"], reverse=True)

print(f"\n{'Rank':<6}{'F1-macro':<22}{'#Feat':<8}{'Features'}")
print("-" * 85)
for rank, r in enumerate(combo_results[:15], 1):
    print(f"{rank:<6}{r['f1_mean']:.4f} +/- {r['f1_std']:.4f}     {r['n_features']:<8}{r['features']}")

best = combo_results[0]
print(f"\nהצירוף המנצח: {best['features']}  (F1-macro: {best['f1_mean']:.4f})")

### הפיצ'רים שנבחרו

הפיצ'רים המנצחים הם: **Stress_Index, GPA, Assignment_Delay_Days, Attendance_Rate**.
צירוף זה נתן את ציון ה-F1-macro הגבוה ביותר בבדיקה על סט האימון.

### גרף 4: תרשים כינור (Violin Plot) – התפלגות הפיצ'רים הנבחרים

Violin Plot משלב בין Box Plot ל-KDE (Kernel Density Estimation). הוא מראה גם את ההתפלגות (הצורה) וגם את הרבעונים. ככה אפשר לראות לא רק את הממוצע והחציון, אלא גם את הצורה המלאה של ההתפלגות – האם היא סימטרית, עם שיא אחד או שניים, וכו'.

In [ ]:
FEATURES = ['Stress_Index', 'GPA', 'Assignment_Delay_Days', 'Attendance_Rate']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    sns.violinplot(x='Dropout', y=col, data=df_cleaned, ax=axes[i], palette="Set2", inner="quartile")
    axes[i].set_title(f'התפלגות {col} לפי סטטוס נשירה', fontsize=12)
    axes[i].set_xlabel('נשירה (0 = לא, 1 = כן)')
    axes[i].set_ylabel(col)

plt.suptitle('Violin Plots – השוואת התפלגויות לפי נשירה', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### גרף 5: תרשים רדאר (Radar Chart) – פרופיל סטודנט

תרשים רדאר מציג מספר משתנים על צירים שונים היוצאים מנקודה מרכזית, כמו קורי עכביש.
הוא מאפשר לראות במבט אחד את ה"פרופיל" של כל קבוצה – נושרים מול ממשיכים.
הערכים מנורמלים (0–1) כדי שאפשר יהיה להשוות בין משתנים בסקאלות שונות.

In [ ]:
scaler_radar = MinMaxScaler()
df_radar = pd.DataFrame(scaler_radar.fit_transform(df_cleaned[FEATURES]), columns=FEATURES)
df_radar['Dropout'] = df_cleaned['Dropout'].values

categories = FEATURES
N = len(categories)

avg_dropout = df_radar[df_radar['Dropout'] == 1][categories].mean().tolist()
avg_graduate = df_radar[df_radar['Dropout'] == 0][categories].mean().tolist()

avg_dropout += avg_dropout[:1]
avg_graduate += avg_graduate[:1]

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.plot(angles, avg_dropout, color='red', linewidth=2, label='נושרים (Dropout=1)')
ax.fill(angles, avg_dropout, color='red', alpha=0.25)

ax.plot(angles, avg_graduate, color='blue', linewidth=2, label='ממשיכים (Dropout=0)')
ax.fill(angles, avg_graduate, color='blue', alpha=0.1)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], categories)

plt.title('פרופיל סטודנט: נושרים מול ממשיכים', size=16, y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.show()

### גרף 6: היסטוגרמה (Histogram + KDE) – התפלגות GPA

היסטוגרמה מראה כמה פעמים כל ערך (או טווח ערכים) מופיע בנתונים. עקומת ה-KDE מוסיפה קו חלק שמייצג את צפיפות ההסתברות. נשווה את ההתפלגות של GPA בין נושרים לממשיכים.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(data=df_cleaned, x='GPA', hue='Dropout', kde=True, bins=30,
             palette={0: 'blue', 1: 'red'}, alpha=0.5, ax=ax)

ax.set_title('התפלגות GPA לפי סטטוס נשירה', fontsize=14)
ax.set_xlabel('GPA')
ax.set_ylabel('מספר סטודנטים')
ax.legend(title='נשירה', labels=['ממשיכים (0)', 'נושרים (1)'])

plt.tight_layout()
plt.show()

### ניתוח שיעור נשירה לפי קבוצות ציונים

נחלק את הסטודנטים ל-3 קבוצות לפי GPA (נמוך, בינוני, גבוה) ונבדוק את שיעור הנשירה בכל קבוצה.
זה מראה אם ציונים נמוכים באמת קשורים לנשירה גבוהה יותר.

In [ ]:
df_plot = df_cleaned.copy()
df_plot['GPA_Group'] = pd.cut(df_plot['GPA'], bins=[0, 2, 3, 4], labels=['נמוך (0-2)', 'בינוני (2-3)', 'גבוה (3-4)'])

dropout_rate = df_plot.groupby('GPA_Group')['Dropout'].mean() * 100
print("שיעור נשירה לפי קבוצת ציונים:")
print(dropout_rate.to_string())

print("\nסטטיסטיקות GPA לפי סטטוס נשירה:")
stats = df_plot.groupby('Dropout')['GPA'].agg(['mean', 'median', 'min', 'max'])
stats.index = ['ממשיכים', 'נושרים']
print(stats.to_string())

print("\nניתוח: ההבדל בין נשירה של קבוצת ציונים נמוכים לבינוניים גדול הרבה יותר")
print("מההפרש בין בינוניים לגבוהים. כלומר, השפעת הציונים על נשירה חזקה במיוחד בציונים נמוכים.")

### תצפיות מהגרפים

מהגרפים ניתן לראות:
- סטודנטים שנשרו הראו ציוני GPA נמוכים משמעותית מאלה שלא נשרו.
- רמת הסטרס (Stress_Index) גבוהה יותר אצל נושרים.
- עיכוב בהגשות (Assignment_Delay_Days) גבוה יותר בקרב נושרים.
- אחוז הנוכחות (Attendance_Rate) נמוך יותר אצל נושרים.
- הפרופיל ברדאר מראה הבדל ברור בין שתי הקבוצות בכל הפרמטרים.

## חלק ג' – בניית מודלים ואימון

### חלוקת הנתונים: Train / Validation / Test

חלוקת הנתונים ל-3 קבוצות חשובה כדי להבטיח שהמודל באמת לומד ולא רק "משנן":
- **Train (60%)** – סט האימון, עליו המודל לומד את הדפוסים.
- **Validation (20%)** – סט האימות, עליו אנחנו מכוונים היפרפרמטרים ובוחרים את המודל הטוב ביותר.
- **Test (20%)** – סט המבחן, משמש **רק בסוף** להערכה סופית. המודל לא רואה אותו במהלך האימון.

אם היינו מחלקים רק ל-train/test, היינו מסתכנים בכך שהכוונון נעשה על סט המבחן – מה שגורם לאומדן אופטימי מדי.

In [ ]:
FEATURES = ['Stress_Index', 'GPA', 'Assignment_Delay_Days', 'Attendance_Rate']
TARGET   = 'Dropout'

x = df_cleaned[FEATURES]
y = df_cleaned[TARGET].values

# First split: 80% train+val, 20% test
x_trainval, x_test, y_trainval, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: 75% of trainval = 60% total for train, 25% of trainval = 20% total for validation
x_train, x_val, y_train, y_val = train_test_split(
    x_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f"גודל סט אימון (Train):    {len(x_train)} ({len(x_train)/len(x)*100:.0f}%)")
print(f"גודל סט אימות (Validation): {len(x_val)} ({len(x_val)/len(x)*100:.0f}%)")
print(f"גודל סט מבחן (Test):       {len(x_test)} ({len(x_test)/len(x)*100:.0f}%)")

print(f"\nהתפלגות Target בכל סט:")
for name, ys in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    counts = pd.Series(ys).value_counts().sort_index()
    print(f"  {name}: {dict(counts)} (נשירה: {counts.get(1,0)/len(ys)*100:.1f}%)")

### איזון הנתונים עם SMOTE

הנתונים לא מאוזנים – יש הרבה יותר סטודנטים שלא נשרו מאלה שנשרו. אם המודל פשוט יחזה "לא נשר" תמיד, הוא יקבל דיוק גבוה אבל לא ישימושי.

**SMOTE** (Synthetic Minority Over-sampling Technique) יוצר דוגמאות סינתטיות של המחלקה המיעוטית כדי לאזן את הנתונים. נבדוק מספר יחסי איזון כדי למצוא את הטוב ביותר.

In [ ]:
cv_smote = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)
smote_strategies = [0.5, 0.6, 0.7, 0.8]
best_smote_ratio = None
best_smote_score = 0.0

for ratio in smote_strategies:
    pipe = ImbPipeline([
        ('smote',  SMOTE(sampling_strategy=ratio, random_state=42)),
        ('scaler', StandardScaler()),
        ('clf',    SVC(kernel='rbf', C=10, random_state=42)),
    ])
    scores = cross_val_score(pipe, x_train, y_train,
                             cv=cv_smote, scoring='f1_macro', n_jobs=-1)
    print(f"  ratio={ratio:.1f}  →  F1-macro: {scores.mean():.4f} ± {scores.std():.4f}")
    if scores.mean() > best_smote_score:
        best_smote_score = scores.mean()
        best_smote_ratio = ratio

if best_smote_ratio is None:
    best_smote_ratio = smote_strategies[0]

print(f"\nהיחס הטוב ביותר ל-SMOTE: {best_smote_ratio} (CV F1-macro: {best_smote_score:.4f})")

### פונקציית הערכה

הפונקציה הבאה מחשבת ומדפיסה את כל מדדי הביצוע של כל מודל:
- **Accuracy** – אחוז הדיוק הכללי.
- **Precision** – מתוך מי שחזינו כנושר, כמה באמת נשרו.
- **Recall** – מתוך כל הנושרים באמת, כמה הצלחנו לזהות.
- **F1-Score** – ממוצע הרמוני של Precision ו-Recall, מתאים לנתונים לא מאוזנים.
- **מטריצת בלבול** – טבלה שמראה כמה חיזויים נכונים ושגויים היו מכל סוג.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    print(f"\n{'=' * 55}")
    print(f"  {name}")
    print(f"{'=' * 55}")

    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")

    prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec_macro  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    prec_w     = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec_w      = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_w       = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"\n{'Metric':<15} {'Macro':<20} {'Weighted'}")
    print(f"{'Precision':<15} {prec_macro:<20.4f} {prec_w:.4f}")
    print(f"{'Recall':<15} {rec_macro:<20.4f} {rec_w:.4f}")
    print(f"{'F1':<15} {f1_macro:<20.4f} {f1_w:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred)).plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix – {name}", fontsize=9)
    plt.tight_layout()
    plt.show()

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_w,
        'precision_macro': prec_macro,
        'precision_weighted': prec_w,
        'recall_macro': rec_macro,
        'recall_weighted': rec_w,
    }

### אימון המודלים

נאמן 3 מודלים שונים, כל אחד עם 2 סוגי נרמול (MinMax ו-Standard):

1. **KNN (K-Nearest Neighbors)** – מסווג לפי השכנים הקרובים ביותר. פשוט ואינטואיטיבי.
2. **SVM (Support Vector Machine)** – מוצא את המישור הטוב ביותר להפרדה בין המחלקות.
3. **MLP (Multi-Layer Perceptron)** – רשת נוירונים שלומדת דפוסים מורכבים.

לכל מודל מבצעים **חקר היפרפרמטרים** (GridSearch / RandomizedSearch) על סט האימון עם Cross-Validation.
**הערכת הביצועים הסופית** תתבצע על סט ה-Validation תחילה, ולבסוף על סט ה-Test.

In [ ]:
# Baseline model
dummy_majority = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_majority.fit(x_train, y_train)
y_pred_dummy = dummy_majority.predict(x_val)
print("=== Baseline (most_frequent) – על סט Validation ===")
baseline_result = evaluate_model("Dummy/majority", y_val, y_pred_dummy)

cv_knn_inner  = StratifiedKFold(n_splits=5,  shuffle=True, random_state=42)
cv_knn_outer  = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_fast_inner = StratifiedKFold(n_splits=3,  shuffle=True, random_state=42)
cv_fast_outer = StratifiedKFold(n_splits=5,  shuffle=True, random_state=42)

scalers = {
    'mn':  MinMaxScaler(),
    'std': StandardScaler(),
}
tuned_results = {}
best_models   = {}

for scaler_name, scaler_obj in scalers.items():

    print(f"\n\n{'#'*70}")
    print(f"  SCALER: {scaler_name.upper()}")
    print(f"{'#'*70}")

    # KNN
    print(f"\n{'='*62}")
    print(f"  KNN — GridSearchCV (inner 5-Fold) / {scaler_name}")
    print(f"{'='*62}")

    knn_pipe = ImbPipeline([
        ('smote',  SMOTE(sampling_strategy=best_smote_ratio, random_state=42)),
        ('scaler', scaler_obj),
        ('clf',    KNeighborsClassifier()),
    ])
    knn_grid = {
        'clf__n_neighbors': [5, 15, 25, 35, 45, 55, 65, 75, 85, 95],
        'clf__weights':     ['uniform', 'distance'],
        'clf__metric':      ['euclidean', 'manhattan'],
    }
    knn_search = GridSearchCV(knn_pipe, knn_grid,
                              cv=cv_knn_inner, scoring='f1_macro',
                              n_jobs=-1, verbose=0)
    knn_search.fit(x_train, y_train)
    y_pred_val = knn_search.best_estimator_.predict(x_val)
    print(f"Best params       : {knn_search.best_params_}")
    print(f"Inner-CV F1-macro : {knn_search.best_score_:.4f}")
    result = evaluate_model(f"KNN / {scaler_name} (Validation)", y_val, y_pred_val)
    outer = cross_val_score(knn_search.best_estimator_, x_train, y_train,
                            cv=cv_knn_outer, scoring='f1_macro', n_jobs=-1)
    print(f"Outer 10-Fold CV  : {outer.mean():.4f} ± {outer.std():.4f}")
    key = f"KNN/{scaler_name}"
    tuned_results[key] = {**result, 'best_params': knn_search.best_params_,
                          'inner_cv_f1': knn_search.best_score_,
                          'outer_cv_f1_mean': outer.mean(), 'outer_cv_f1_std': outer.std()}
    best_models[key] = knn_search.best_estimator_

    # SVM
    print(f"\n{'='*62}")
    print(f"  SVM — RandomizedSearchCV (inner 3-Fold, n_iter=10) / {scaler_name}")
    print(f"{'='*62}")

    svm_pipe = ImbPipeline([
        ('smote',  SMOTE(sampling_strategy=best_smote_ratio, random_state=42)),
        ('scaler', scaler_obj),
        ('clf',    SVC(random_state=42)),
    ])
    svm_dist = {
        'clf__C':      [1, 10, 100],
        'clf__kernel': ['linear', 'rbf'],
        'clf__gamma':  ['scale', 'auto'],
    }
    svm_search = RandomizedSearchCV(svm_pipe, svm_dist, n_iter=10,
                                    cv=cv_fast_inner, scoring='f1_macro',
                                    n_jobs=-1, random_state=42, verbose=0)
    svm_search.fit(x_train, y_train)
    y_pred_val = svm_search.best_estimator_.predict(x_val)
    print(f"Best params       : {svm_search.best_params_}")
    print(f"Inner-CV F1-macro : {svm_search.best_score_:.4f}")
    result = evaluate_model(f"SVM / {scaler_name} (Validation)", y_val, y_pred_val)
    outer = cross_val_score(svm_search.best_estimator_, x_train, y_train,
                            cv=cv_fast_outer, scoring='f1_macro', n_jobs=-1)
    print(f"Outer 5-Fold CV   : {outer.mean():.4f} ± {outer.std():.4f}")
    key = f"SVM/{scaler_name}"
    tuned_results[key] = {**result, 'best_params': svm_search.best_params_,
                          'inner_cv_f1': svm_search.best_score_,
                          'outer_cv_f1_mean': outer.mean(), 'outer_cv_f1_std': outer.std()}
    best_models[key] = svm_search.best_estimator_

    # MLP
    print(f"\n{'='*62}")
    print(f"  MLP — RandomizedSearchCV (inner 3-Fold, n_iter=10) / {scaler_name}")
    print(f"{'='*62}")

    mlp_pipe = ImbPipeline([
        ('smote',  SMOTE(sampling_strategy=best_smote_ratio, random_state=42)),
        ('scaler', scaler_obj),
        ('clf',    MLPClassifier(max_iter=200, early_stopping=True, random_state=42)),
    ])
    mlp_dist = {
        'clf__hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
        'clf__activation':         ['relu', 'tanh'],
        'clf__learning_rate_init': [0.001, 0.01],
    }
    mlp_search = RandomizedSearchCV(mlp_pipe, mlp_dist, n_iter=10,
                                    cv=cv_fast_inner, scoring='f1_macro',
                                    n_jobs=-1, random_state=42, verbose=0)
    mlp_search.fit(x_train, y_train)
    y_pred_val = mlp_search.best_estimator_.predict(x_val)
    print(f"Best params       : {mlp_search.best_params_}")
    print(f"Inner-CV F1-macro : {mlp_search.best_score_:.4f}")
    result = evaluate_model(f"MLP / {scaler_name} (Validation)", y_val, y_pred_val)
    outer = cross_val_score(mlp_search.best_estimator_, x_train, y_train,
                            cv=cv_fast_outer, scoring='f1_macro', n_jobs=-1)
    print(f"Outer 5-Fold CV   : {outer.mean():.4f} ± {outer.std():.4f}")
    key = f"MLP/{scaler_name}"
    tuned_results[key] = {**result, 'best_params': mlp_search.best_params_,
                          'inner_cv_f1': mlp_search.best_score_,
                          'outer_cv_f1_mean': outer.mean(), 'outer_cv_f1_std': outer.std()}
    best_models[key] = mlp_search.best_estimator_

### טבלת השוואת תוצאות (Validation Set)

להלן סיכום של כל המודלים שנבדקו. הטבלה מציגה את הדיוק, F1-macro, ותוצאות ה-Cross-Validation.

In [ ]:
print("=" * 90)
print("סיכום תוצאות כל המודלים (על סט Validation)")
print("=" * 90)
print(f"{'מודל':<22}{'Accuracy':<16}{'F1-Macro':<16}{'CV F1'}")
print("-" * 90)
for mn, r in tuned_results.items():
    cv_str = f"{r['outer_cv_f1_mean']:.4f} ± {r['outer_cv_f1_std']:.4f}"
    print(f"{mn:<22}{r['accuracy']:<16.4f}{r['f1_macro']:<16.4f}{cv_str}")

best_key = max(tuned_results, key=lambda k: tuned_results[k]['f1_macro'])
print(f"\n{'=' * 60}")
print(f"המודל המנצח (לפי Validation): {best_key}")
print(f"F1-macro על Validation: {tuned_results[best_key]['f1_macro']:.4f}")
print(f"היפרפרמטרים: {tuned_results[best_key]['best_params']}")
print(f"{'=' * 60}")

### הערכה סופית על סט המבחן (Test Set)

לאחר שבחרנו את המודל הטוב ביותר על סמך ביצועיו על סט ה-Validation, נבדוק אותו **פעם אחת בלבד** על סט ה-Test.
זה נותן לנו הערכה אמינה של ביצועי המודל על נתונים שהוא מעולם לא ראה.

In [ ]:
print(f"הערכה סופית של המודל המנצח ({best_key}) על סט ה-Test:")
print("=" * 60)

best_model = best_models[best_key]
y_pred_test = best_model.predict(x_test)
final_result = evaluate_model(f"{best_key} (Test Set – הערכה סופית)", y_test, y_pred_test)

print(f"\nהשוואה:")
print(f"  F1-macro על Validation: {tuned_results[best_key]['f1_macro']:.4f}")
print(f"  F1-macro על Test:       {final_result['f1_macro']:.4f}")
diff = abs(tuned_results[best_key]['f1_macro'] - final_result['f1_macro'])
if diff < 0.05:
    print(f"  הפער ({diff:.4f}) קטן – המודל מכליל היטב!")
else:
    print(f"  הפער ({diff:.4f}) מעיד על אפשרות של overfitting.")

## חלק ד' – ניתוח תוצאות ומסקנות

### ניתוח תוצאות המודלים

בבלוק הבא נציג ניתוח מפורט של התוצאות: איזה מודל ניצח, למה הוא ניצח, ומה המשמעות של כל מדד.

In [ ]:
print("=" * 70)
print("ניתוח מפורט של תוצאות המודלים")
print("=" * 70)

results_df = pd.DataFrame(tuned_results).T
results_df = results_df.sort_values('f1_macro', ascending=False)

print("\nדירוג המודלים לפי F1-Macro:")
print("-" * 50)
for rank, (name, row) in enumerate(results_df.iterrows(), 1):
    print(f"  {rank}. {name}: F1-macro={row['f1_macro']:.4f}, Accuracy={row['accuracy']:.4f}")

print(f"\n\nהמודל המנצח: {best_key}")
print(f"\nלמה הוא המנצח?")
print(f"  - F1-macro הגבוה ביותר ({tuned_results[best_key]['f1_macro']:.4f}) – ")
print(f"    מדד זה חשוב כי הנתונים לא מאוזנים, ו-F1 מביא בחשבון גם Precision וגם Recall.")
print(f"  - Cross-Validation יציב ({tuned_results[best_key]['outer_cv_f1_mean']:.4f} ± {tuned_results[best_key]['outer_cv_f1_std']:.4f}) – ")
print(f"    סטיית תקן נמוכה מעידה שהמודל עקבי ולא תלוי בחלוקה ספציפית.")

print(f"\nניתוח לפי סוג מודל:")
for model_type in ['KNN', 'SVM', 'MLP']:
    keys = [k for k in tuned_results if k.startswith(model_type)]
    if keys:
        best_of_type = max(keys, key=lambda k: tuned_results[k]['f1_macro'])
        r = tuned_results[best_of_type]
        print(f"\n  {model_type} (הגרסה הטובה ביותר: {best_of_type}):")
        print(f"    Accuracy: {r['accuracy']:.4f}")
        print(f"    F1-macro: {r['f1_macro']:.4f}")
        print(f"    Precision: {r['precision_macro']:.4f}")
        print(f"    Recall: {r['recall_macro']:.4f}")

print(f"\nניתוח לפי סוג נרמול:")
for scaler_type in ['mn', 'std']:
    keys = [k for k in tuned_results if k.endswith(scaler_type)]
    if keys:
        avg_f1 = np.mean([tuned_results[k]['f1_macro'] for k in keys])
        print(f"  {scaler_type.upper()}: ממוצע F1-macro = {avg_f1:.4f}")

### מסקנות על נשירת סטודנטים

על סמך הנתונים, הגרפים, ותוצאות המודלים, להלן המסקנות העיקריות:

**1. הגורמים המשפיעים ביותר על נשירה:**
- **מדד סטרס (Stress_Index)** – סטודנטים עם רמת סטרס גבוהה נוטים לנשור יותר. זהו ככל הנראה הגורם המשפיע ביותר.
- **ציון ממוצע (GPA)** – ציונים נמוכים מנבאים נשירה. ההשפעה חזקה במיוחד בציונים מתחת ל-2.
- **עיכוב בהגשות (Assignment_Delay_Days)** – סטודנטים שמגישים מאוחר נמצאים בסיכון גבוה יותר.
- **אחוז נוכחות (Attendance_Rate)** – נוכחות נמוכה קשורה לנשירה.

**2. פרופיל סטודנט בסיכון:**
סטודנט עם GPA נמוך, סטרס גבוה, איחורים בהגשות, ונוכחות נמוכה – נמצא בסיכון גבוה מאוד לנשירה.

**3. המלצות מעשיות:**
- זיהוי מוקדם של סטודנטים בסיכון על סמך הפרמטרים הללו.
- הפעלת תוכניות תמיכה ממוקדות: ייעוץ אקדמי, סדנאות ניהול סטרס, ומעקב אחר הגשות.
- התמקדות בסטודנטים עם GPA נמוך – שם ההתערבות יכולה להיות הכי אפקטיבית.

**4. מגבלות המחקר:**
- המאגר מבוסס על סימולציה ולא על נתונים אמיתיים, לכן המסקנות הן אינדיקטיביות בלבד.
- אין מידע על גורמים נוספים כמו מצב כלכלי מפורט, תמיכה משפחתית, או מוטיבציה.

## סיכום הפרויקט

בפרויקט זה בנינו מערכת לחיזוי נשירת סטודנטים מאוניברסיטה. עברנו את כל שלבי מדעי הנתונים:

1. **טעינה וסקירה** – טענו את המאגר, בדקנו את המבנה, הסטטיסטיקות, הכפילויות, והערכים החסרים.
2. **ניקוי ועיבוד** – המרנו משתנים קטגוריאליים למספרים, טיפלנו בחריגים ובערכים חסרים.
3. **ויזואליזציה** – יצרנו 6 סוגי גרפים שונים (מפה תרמית, עמודות, קופסה, כינור, רדאר, היסטוגרמה) כדי להבין את הנתונים ולזהות דפוסים.
4. **בחירת פיצ'רים** – בדקנו את כל הצירופים האפשריים ומצאנו את 4 הפיצ'רים הטובים ביותר.
5. **בניית מודלים** – אימנו 3 מודלים (KNN, SVM, MLP) עם 2 סוגי נרמול וחקר היפרפרמטרים.
6. **הערכה** – חילקנו ל-train/validation/test, השוונו ביצועים, ובחרנו את המודל המנצח.

**התוצאה:** הצלחנו לבנות מודל שמזהה סטודנטים בסיכון נשירה ברמת דיוק טובה, כפי שמעיד ציון ה-F1-macro.

## רפלקציה אישית

הפרויקט הזה לימד אותי המון על תהליך העבודה במדעי הנתונים. הנה כמה דברים שלמדתי:

**מה למדתי:**
- איך לנקות ולעבד נתונים – הבנתי שרוב העבודה במדעי הנתונים היא לא בבניית המודל, אלא בהכנת הנתונים.
- למה חשוב להמיר משתנים קטגוריאליים ולטפל בחריגים לפני שמכניסים נתונים למודל.
- איך לבחור את הפיצ'רים הטובים ביותר ולמה זה משפיע כל כך על התוצאות.
- למה חלוקה ל-3 קבוצות (train/validation/test) חשובה יותר מחלוקה ל-2.
- איך להשתמש ב-SMOTE כדי להתמודד עם נתונים לא מאוזנים.

**אתגרים שנתקלתי בהם:**
- הבנת מדדי הביצוע השונים (Accuracy, Precision, Recall, F1) – בהתחלה היה מבלבל, אבל עם הזמן הבנתי שכל מדד מודד דבר שונה.
- בחירת ההיפרפרמטרים – יש כל כך הרבה אפשרויות, והבנתי למה חשוב לעשות חיפוש שיטתי (GridSearch) במקום לנחש.
- הטיפול בחוסר האיזון בנתונים – למדתי שדיוק גבוה לא תמיד אומר שהמודל טוב.

**מה הייתי משפר:**
- הייתי מנסה עוד מודלים כמו Random Forest ו-XGBoost.
- הייתי בודק יותר פיצ'רים ואולי יוצר פיצ'רים חדשים (Feature Engineering).
- הייתי מנסה להשתמש בנתונים אמיתיים ולא בסימולציה.

**לסיום,** הפרויקט הראה לי שמדעי הנתונים זה לא רק "להריץ קוד" – אלא תהליך של חשיבה, ניתוח, והבנה עמוקה של הנתונים.